# Build a Language Model From Scratch: GuppyLM

In this notebook you will train a small language model (LLM) that talks like a fish.
You will see every step: the data, the tokenizer, the model, the training loop, and text generation.

GuppyLM has about 9 million parameters. GPT-style models with billions of parameters work the same way, only bigger.
So if you understand this notebook, you understand the core of how a chat model is built.

**How to run it**
1. Open the menu **Runtime > Change runtime type** and pick a GPU (see the note below).
2. Run the cells from top to bottom with **Shift+Enter**. Read the text above each cell first.
3. Total time: about 6 minutes on an L4 GPU (training itself takes about 4.5 minutes).

**Which GPU?** This model is tiny: 8.7 million parameters and batches of 32 short sentences. It needs very little GPU memory, and a bigger GPU helps less than you might expect, because a lot of each training step goes to Python overhead and launching many small GPU jobs.
An **L4** is a good pick. It is fast enough (10,000 training steps in about 4.5 minutes) and it uses far fewer Colab compute units per hour than an A100 or H100.
A free **T4** also works.

**Map of this notebook**

| Step | What you do | Idea you learn |
|---|---|---|
| 1 | Check the GPU | Why GPUs matter |
| 2 | Write the source files | Where the code lives |
| 3 | Look at the data | A model learns from examples |
| 4 | Train a tokenizer | Text becomes numbers |
| 5 | Build the model | Embeddings, attention, tensor shapes |
| 6 | Train on one batch | How learning works (loss, gradients) |
| 7 | Train for real | Learning rate schedule, loss curve |
| 8 | Talk to Guppy | Sampling and temperature |
| 9 | Experiments | Change something and watch what happens |

## 1. Setup and GPU check

We install the libraries and make a clean working folder. Then we check which device PyTorch can use.

A language model is mostly big matrix multiplications. A GPU runs thousands of small multiplications at the same time, so it is much faster than a CPU for this work.

In [ ]:
!pip install -q tokenizers datasets

import os, shutil, time
import torch

# Start from an empty folder so old files from earlier runs cannot confuse us.
if os.path.exists('/content/guppy'):
    shutil.rmtree('/content/guppy')
os.makedirs('/content/guppy')
os.chdir('/content/guppy')

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU found. Use Runtime > Change runtime type, then pick a GPU. Training on CPU takes much longer.')

Let's measure the difference. The cell below multiplies two 2048 x 2048 matrices ten times, on the CPU and then on the GPU.

In [ ]:
def time_matmul(device, n=2048, repeats=10):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    a @ b                                   # warm-up run, not timed
    if device == 'cuda':
        torch.cuda.synchronize()            # GPU work is asynchronous: wait for it to finish
    t0 = time.time()
    for _ in range(repeats):
        a @ b
    if device == 'cuda':
        torch.cuda.synchronize()
    return (time.time() - t0) / repeats

cpu_t = time_matmul('cpu')
print(f'CPU: {cpu_t*1000:.1f} ms per multiplication')
if torch.cuda.is_available():
    gpu_t = time_matmul('cuda')
    print(f'GPU: {gpu_t*1000:.1f} ms per multiplication  ({cpu_t/gpu_t:.0f}x faster)')

## 2. The source files

The next five cells write the real GuppyLM code to disk. Each file has one job.
You do not need to understand every line now. Each section below points back to the part that matters.

| File | Job |
|---|---|
| `config.py` | All the knobs: model size, learning rate, number of steps |
| `model.py` | The transformer itself |
| `dataset.py` | Turns text into batches of numbers |
| `train.py` | The training loop |
| `inference.py` | Loads a trained model and generates replies |

`%%writefile` is a Colab trick: it saves the cell's content as a file instead of running it.

**`config.py`**. Two small classes that only hold numbers. Look at `GuppyConfig`: `d_model=384` is the width of every vector inside the model, `n_layers=6` is how many transformer blocks are stacked, and `vocab_size=4096` is how many different tokens exist.

In [ ]:
%%writefile config.py
"""GuppyLM configuration."""

from dataclasses import dataclass


@dataclass
class GuppyConfig:
    vocab_size: int = 4096
    max_seq_len: int = 128
    d_model: int = 384
    n_layers: int = 6
    n_heads: int = 6
    ffn_hidden: int = 768
    dropout: float = 0.1

    # Special tokens
    pad_id: int = 0
    bos_id: int = 1           # <|im_start|>
    eos_id: int = 2           # <|im_end|>


@dataclass
class TrainConfig:
    batch_size: int = 32
    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    warmup_steps: int = 200
    max_steps: int = 10000
    eval_interval: int = 200
    save_interval: int = 500
    grad_clip: float = 1.0
    device: str = "auto"
    seed: int = 42
    data_dir: str = "data"
    output_dir: str = "checkpoints"

**`model.py`**. The heart of the project, about 130 lines. We take it apart in step 5, so just skim it now.

In [ ]:
%%writefile model.py
"""
GuppyLM — a tiny fish brain.

Vanilla transformer: multi-head attention, ReLU FFN, LayerNorm, learned positional embeddings.
No GQA, no SwiGLU, no parallel residual, no RoPE. As simple as it gets.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import GuppyConfig


class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.out = nn.Linear(config.d_model, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float("-inf"))
        attn = self.dropout(F.softmax(attn, dim=-1))
        return self.out((attn @ v).transpose(1, 2).contiguous().view(B, T, C))


class FFN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.up = nn.Linear(config.d_model, config.ffn_hidden)
        self.down = nn.Linear(config.ffn_hidden, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.down(F.relu(self.up(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.d_model)
        self.attn = Attention(config)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.ffn = FFN(config)

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class GuppyLM(nn.Module):
    def __init__(self, config: GuppyConfig):
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.norm = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # tie weights

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)

        for block in self.blocks:
            x = block(x, mask)

        logits = self.lm_head(self.norm(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                targets.view(-1),
                ignore_index=0,
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=64, temperature=0.7, top_k=50, **kwargs):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
            if next_id.item() == self.config.eos_id:
                break
        return idx, []

    def param_count(self):
        total = sum(p.numel() for p in self.parameters())
        return total, 0

    def param_summary(self):
        total, _ = self.param_count()
        return f"GuppyLM: {total:,} params ({total/1e6:.1f}M)"

**`dataset.py`**. Notice `x = ids[:-1]` and `y = ids[1:]`. The target `y` is the input `x` shifted by one token. That one line is the whole training task: **predict the next token**.

In [ ]:
%%writefile dataset.py
"""GuppyLM dataset loading."""

import json

import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer


class GuppyDataset(Dataset):
    def __init__(self, path: str, tokenizer_path: str, max_len: int = 512):
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.max_len = max_len
        self.samples = []

        with open(path) as f:
            for line in f:
                data = json.loads(line)
                ids = self.tokenizer.encode(data["text"]).ids
                if len(ids) > max_len:
                    ids = ids[:max_len]
                if len(ids) >= 2:
                    self.samples.append(ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        x = ids[:-1]
        y = ids[1:]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


def collate_fn(batch, pad_id=0):
    xs, ys = zip(*batch)
    max_len = max(len(x) for x in xs)
    padded_x = torch.full((len(xs), max_len), pad_id, dtype=torch.long)
    padded_y = torch.full((len(ys), max_len), pad_id, dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        padded_x[i, :len(x)] = x
        padded_y[i, :len(y)] = y
    return padded_x, padded_y


def get_dataloader(path, tokenizer_path, max_len=512, batch_size=32, shuffle=True):
    dataset = GuppyDataset(path, tokenizer_path, max_len)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        collate_fn=collate_fn, num_workers=0, pin_memory=True,
    )

**`train.py`**. The loop that repeats: get a batch, compute the loss, compute gradients, update the weights. We run it in step 7.

In [ ]:
%%writefile train.py
"""GuppyLM training loop."""

import json
import math
import os
import time

import torch

from config import GuppyConfig, TrainConfig
from dataset import get_dataloader
from model import GuppyLM


def get_device(config):
    if config.device == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(config.device)


def get_lr(step, config):
    if step < config.warmup_steps:
        return config.learning_rate * step / config.warmup_steps
    progress = (step - config.warmup_steps) / max(1, config.max_steps - config.warmup_steps)
    coeff = 0.5 * (1 + math.cos(math.pi * progress))
    return config.min_lr + (config.learning_rate - config.min_lr) * coeff


@torch.no_grad()
def evaluate(model, loader, device, max_batches=50):
    model.eval()
    total_loss, n = 0, 0
    for x, y in loader:
        if n >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item()
        n += 1
    model.train()
    return total_loss / max(1, n)


def train():
    mc = GuppyConfig()
    tc = TrainConfig()
    device = get_device(tc)
    torch.manual_seed(tc.seed)

    print(f"Device: {device}")

    tokenizer_path = os.path.join(tc.data_dir, "tokenizer.json")
    model = GuppyLM(mc).to(device)
    print(model.param_summary())

    train_loader = get_dataloader(
        os.path.join(tc.data_dir, "train.jsonl"), tokenizer_path,
        mc.max_seq_len, tc.batch_size, shuffle=True,
    )
    eval_loader = get_dataloader(
        os.path.join(tc.data_dir, "eval.jsonl"), tokenizer_path,
        mc.max_seq_len, tc.batch_size, shuffle=False,
    )
    print(f"Train: {len(train_loader.dataset):,}, Eval: {len(eval_loader.dataset):,}")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=tc.learning_rate,
        weight_decay=tc.weight_decay, betas=(0.9, 0.95),
    )

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    os.makedirs(tc.output_dir, exist_ok=True)
    with open(os.path.join(tc.output_dir, "config.json"), "w") as f:
        json.dump({"model": vars(mc), "train": vars(tc)}, f, indent=2)

    model.train()
    step, best_eval = 0, float("inf")
    losses = []
    t0 = time.time()

    print(f"\nTraining for {tc.max_steps} steps...")
    print(f"{'Step':>6} | {'LR':>10} | {'Train':>10} | {'Eval':>10} | {'Time':>8}")
    print("-" * 56)

    while step < tc.max_steps:
        for x, y in train_loader:
            if step >= tc.max_steps:
                break

            x, y = x.to(device), y.to(device)
            lr = get_lr(step, tc)
            for pg in optimizer.param_groups:
                pg["lr"] = lr

            if use_amp:
                with torch.amp.autocast("cuda"):
                    _, loss = model(x, y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                _, loss = model(x, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())

            if step % 100 == 0:
                avg = sum(losses[-100:]) / len(losses[-100:])
                elapsed = time.time() - t0
                print(f"{step:6d} | {lr:10.6f} | {avg:10.4f} | {'--':>10} | {elapsed:7.1f}s")

            if step > 0 and step % tc.eval_interval == 0:
                el = evaluate(model, eval_loader, device)
                avg_train = sum(losses[-tc.eval_interval:]) / min(len(losses), tc.eval_interval)
                elapsed = time.time() - t0
                print(f"{step:6d} | {lr:10.6f} | {avg_train:10.4f} | {el:10.4f} | {elapsed:7.1f}s")

                if el < best_eval:
                    best_eval = el
                    torch.save({
                        "step": step,
                        "model_state_dict": model.state_dict(),
                        "config": vars(mc),
                        "eval_loss": el,
                    }, os.path.join(tc.output_dir, "best_model.pt"))
                    print(f"  -> Best model (eval={el:.4f})")

            if step > 0 and step % tc.save_interval == 0:
                torch.save({
                    "step": step,
                    "model_state_dict": model.state_dict(),
                    "config": vars(mc),
                }, os.path.join(tc.output_dir, f"step_{step}.pt"))

            step += 1

    # Final save
    torch.save({
        "step": step,
        "model_state_dict": model.state_dict(),
        "config": vars(mc),
        "train_losses": losses,
    }, os.path.join(tc.output_dir, "final_model.pt"))

    elapsed = time.time() - t0
    print(f"\nDone! {elapsed:.0f}s, best eval: {best_eval:.4f}")


if __name__ == "__main__":
    train()

**`inference.py`**. Loads a saved model and turns a user message into a reply.

In [ ]:
%%writefile inference.py
"""GuppyLM inference — simple chat."""

import json
import os
import time
import uuid

import torch
from tokenizers import Tokenizer

from config import GuppyConfig
from model import GuppyLM


class GuppyInference:
    def __init__(self, checkpoint_path, tokenizer_path, device="cpu"):
        self.device = torch.device(device)
        self.tokenizer = Tokenizer.from_file(tokenizer_path)

        ckpt = torch.load(checkpoint_path, map_location=self.device, weights_only=False)

        # Load config.json from same directory as the model file
        config_dir = os.path.dirname(os.path.abspath(checkpoint_path))
        config_path = os.path.join(config_dir, "config.json")

        # Extract state_dict — handle both legacy and standard formats
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        else:
            state_dict = ckpt

        # Load config — try config.json first, fall back to embedded config
        if os.path.exists(config_path):
            with open(config_path) as f:
                cfg = json.load(f)
            # Support both HF standard keys and our own keys
            self.config = GuppyConfig(
                vocab_size=cfg.get("vocab_size", 4096),
                max_seq_len=cfg.get("max_position_embeddings", cfg.get("max_seq_len", 128)),
                d_model=cfg.get("hidden_size", cfg.get("d_model", 384)),
                n_layers=cfg.get("num_hidden_layers", cfg.get("n_layers", 6)),
                n_heads=cfg.get("num_attention_heads", cfg.get("n_heads", 6)),
                ffn_hidden=cfg.get("intermediate_size", cfg.get("ffn_hidden", 768)),
                dropout=cfg.get("hidden_dropout_prob", cfg.get("dropout", 0.1)),
                pad_id=cfg.get("pad_token_id", cfg.get("pad_id", 0)),
                bos_id=cfg.get("bos_token_id", cfg.get("bos_id", 1)),
                eos_id=cfg.get("eos_token_id", cfg.get("eos_id", 2)),
            )
        elif isinstance(ckpt, dict) and "config" in ckpt:
            valid_fields = {f.name for f in GuppyConfig.__dataclass_fields__.values()}
            self.config = GuppyConfig(**{k: v for k, v in ckpt["config"].items() if k in valid_fields})
        else:
            print("Warning: No config found, using defaults")
            self.config = GuppyConfig()

        self.model = GuppyLM(self.config).to(self.device)
        filtered = {k: v for k, v in state_dict.items() if k in self.model.state_dict()}
        self.model.load_state_dict(filtered)
        self.model.eval()

        total, _ = self.model.param_count()
        print(f"GuppyLM loaded: {total/1e6:.1f}M params")

    def chat_completion(self, messages, temperature=0.7, max_tokens=64,
                        top_k=50, **kwargs):
        """Chat completion — takes messages, returns response."""
        prompt = self._format_prompt(messages)
        input_ids = self.tokenizer.encode(prompt).ids
        prompt_tokens = len(input_ids)
        input_t = torch.tensor([input_ids], dtype=torch.long, device=self.device)

        output_t, _ = self.model.generate(input_t, max_tokens, temperature, top_k)
        output_text = self.tokenizer.decode(output_t[0].tolist()[prompt_tokens:])
        # Truncate at first <|im_end|> — don't let the model leak into the next turn
        if "<|im_end|>" in output_text:
            output_text = output_text.split("<|im_end|>")[0]
        # Also strip any <|im_start|> fragments
        if "<|im_start|>" in output_text:
            output_text = output_text.split("<|im_start|>")[0]
        resp_text = output_text.strip()

        return {
            "choices": [{
                "message": {"role": "assistant", "content": resp_text},
            }],
        }

    def _format_prompt(self, messages):
        parts = []
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content") or ""
            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
        parts.append("<|im_start|>assistant\n")
        return "\n".join(parts)


def main():
    import argparse
    p = argparse.ArgumentParser(description="Chat with Guppy")
    p.add_argument("--checkpoint", default="checkpoints/best_model.pt")
    p.add_argument("--tokenizer", default="data/tokenizer.json")
    p.add_argument("--device", default="cpu")
    p.add_argument("--prompt", "-p", help="Single prompt mode: ask one question and exit")
    args = p.parse_args()

    engine = GuppyInference(args.checkpoint, args.tokenizer, args.device)

    if args.prompt:
        result = engine.chat_completion([{"role": "user", "content": args.prompt}])
        print(result["choices"][0]["message"]["content"])
        return

    print("\nGuppy Chat (type 'quit' to exit)")
    while True:
        inp = input("\nYou> ").strip()
        if inp.lower() in ("quit", "exit", "q"):
            break
        result = engine.chat_completion([{"role": "user", "content": inp}])
        msg = result["choices"][0]["message"]
        if msg.get("content"):
            print(f"Guppy> {msg['content']}")


if __name__ == "__main__":
    main()

## 3. The data

A language model learns from examples. GuppyLM uses 60,000 short conversations between a user and a fish, written by a script and shared on HuggingFace.

We save each conversation in **ChatML** format, a plain-text layout many chat models use:

```
<|im_start|>user
hi guppy<|im_end|>
<|im_start|>assistant
hello. the water is nice today.<|im_end|>
```

`<|im_start|>` and `<|im_end|>` are special markers. They tell the model where a message starts and ends. The model learns to write `<|im_end|>` when its answer is finished.

In [ ]:
import json, random
from collections import Counter
from datasets import load_dataset

ds = load_dataset('arman-bd/guppylm-60k-generic')
print(ds)

os.makedirs('data', exist_ok=True)
texts = []
for split, path in [('train', 'data/train.jsonl'), ('test', 'data/eval.jsonl')]:
    with open(path, 'w') as f:
        for row in ds[split]:
            text = (
                f'<|im_start|>user\n{row["input"]}<|im_end|>\n'
                f'<|im_start|>assistant\n{row["output"]}<|im_end|>'
            )
            f.write(json.dumps({'text': text, 'category': row['category']}) + '\n')
            texts.append(text)
    print(f'{path}: {len(ds[split]):,} conversations')

print('\nFive random examples:\n')
random.seed(0)
for row in random.sample(list(ds['train']), 5):
    print(f'[{row["category"]}]')
    print(f'  user:   {row["input"]}')
    print(f'  guppy:  {row["output"]}\n')

cats = Counter(ds['train']['category'])
print(f'{len(cats)} topics, for example: {cats.most_common(5)}')

## 4. The tokenizer: text to numbers

A neural network only understands numbers. A **tokenizer** cuts text into pieces called tokens and gives each piece an integer id.

We use **BPE** (byte-pair encoding). It starts from single characters and keeps merging the pairs that appear most often in the training text. Common words end up as one token, rare words get split into several pieces.

We ask for a vocabulary of up to 4,096 tokens. Three of them are reserved for special markers: `<pad>` (id 0, filler to make batches the same length), `<|im_start|>` (id 1) and `<|im_end|>` (id 2).

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=4096,
    special_tokens=['<pad>', '<|im_start|>', '<|im_end|>'],
    min_frequency=2,
    show_progress=False,
)
tokenizer.train_from_iterator(texts, trainer)
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
tokenizer.save('data/tokenizer.json')
print(f'Vocabulary size: {tokenizer.get_vocab_size()}')

**Why is the vocabulary smaller than 4,096?** The fish uses a small, repetitive set of words. The trainer stops merging when no pair of pieces appears at least twice, so it ends with fewer tokens than the limit. The model still allocates 4,096 slots, and the unused ones just stay untrained. That is why the loss check in step 5 uses 4,096.

Now play with it. Change the sentence and run the cell again. Words the fish knows well (water, food) become one token. Words it has never seen, like "quantum", split into smaller pieces.
The `Ġ` you see at the start of a token is how this tokenizer writes a leading space, so `Ġlove` means " love".

In [ ]:
sentence = 'i love food and the water is warm'   # <- try your own sentence

enc = tokenizer.encode(sentence)
print('tokens:', enc.tokens)
print('ids:   ', enc.ids)
print('decoded back:', tokenizer.decode(enc.ids))

print('\nA word the fish never saw:')
enc2 = tokenizer.encode('quantum')
print(enc2.tokens)

## 5. The model

Now we build the model and follow one batch of data through it, looking at the **shape** of the tensor at each stage.
A tensor is a multi-dimensional array (like a NumPy array that can also live on the GPU). Shapes are the easiest way to understand what a network does.

We use these letters:
- `B`: batch size, how many sentences we process at once
- `T`: sequence length, how many tokens in each sentence
- `C`: the model width (`d_model`, 384 here)

In [ ]:
import torch.nn.functional as F
from config import GuppyConfig
from model import GuppyLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = GuppyConfig()
model = GuppyLM(config).to(device)
print(model.param_summary())
print(f'{config.n_layers} layers, {config.n_heads} attention heads, width {config.d_model}')

# Take a real sentence, turn it into ids, and add a batch dimension.
prompt = '<|im_start|>user\nhi guppy<|im_end|>\n<|im_start|>assistant\n'
ids = tokenizer.encode(prompt).ids
x = torch.tensor([ids], device=device)
print(f'\nInput ids shape (B, T): {tuple(x.shape)}')

model.eval()
with torch.no_grad():
    B, T = x.shape
    tok = model.tok_emb(x)
    print(f'1. Token embedding  (B, T, C): {tuple(tok.shape)}   <- each id becomes a vector of {config.d_model} numbers')
    pos = model.pos_emb(torch.arange(T, device=device))
    print(f'2. Position embedding (T, C):  {tuple(pos.shape)}   <- tells the model where each token sits')
    h = tok + pos
    mask = torch.tril(torch.ones(T, T, device=device)).unsqueeze(0).unsqueeze(0)
    for i, block in enumerate(model.blocks):
        h = block(h, mask)
    print(f'3. After {config.n_layers} blocks (B, T, C): {tuple(h.shape)}   <- same shape, richer meaning')
    logits = model.lm_head(model.norm(h))
    print(f'4. Logits (B, T, vocab): {tuple(logits.shape)}   <- one score per possible next token, at every position')

**What happens inside a block?** Each of the 6 blocks does two things:

1. **Attention.** Every token looks at the earlier tokens and decides which ones matter for it. The model has 6 attention heads, so it can look for 6 different kinds of relationships at once.
2. **Feed-forward network (FFN).** A small two-layer network that processes each token on its own.

Both steps use a **residual connection** (`x = x + ...`): the block adds its result to the input instead of replacing it. This keeps information flowing and makes deep networks easier to train.

**The causal mask.** When the model predicts token 5, it must not peek at token 6. A triangular mask hides the future. Here is what it looks like for 6 tokens (1 means "can look at"):

In [ ]:
print(torch.tril(torch.ones(6, 6)).int())

**A sanity check.** The model is untrained, so its guesses are random. With 4,096 possible tokens, a random guess gives the right token with probability 1/4096.
The loss (cross-entropy) for that is `ln(4096)`, about 8.32. If your first loss is near this number, the setup is correct.

In [ ]:
import math
from dataset import get_dataloader

eval_loader = get_dataloader('data/eval.jsonl', 'data/tokenizer.json', config.max_seq_len, batch_size=32, shuffle=False)
x_batch, y_batch = next(iter(eval_loader))
x_batch, y_batch = x_batch.to(device), y_batch.to(device)
print('Batch shapes (B, T):', tuple(x_batch.shape), tuple(y_batch.shape))

model.eval()
with torch.no_grad():
    _, loss = model(x_batch, y_batch)
print(f'Loss of the untrained model: {loss.item():.3f}')
print(f'ln(4096) = {math.log(4096):.3f}')

## 6. How learning works: overfit one batch

Training repeats four steps:

1. **Forward pass.** Run the batch through the model and get a loss, a single number that says how wrong the predictions are.
2. **Backward pass.** `loss.backward()` computes, for every weight, how much a small change would change the loss. These are the **gradients**.
3. **Update.** The optimizer (AdamW) nudges each weight a little in the direction that lowers the loss.
4. **Reset.** Clear the gradients and repeat.

Before the full run, let's watch this on a single batch. We train on the same 32 sentences again and again. The loss should fall fast, because the model can simply memorise them. (This is called **overfitting**. It is a useful debugging trick: if a model cannot even memorise one batch, something is broken.)

In [ ]:
torch.manual_seed(0)
tiny_model = GuppyLM(config).to(device)
opt = torch.optim.AdamW(tiny_model.parameters(), lr=1e-3)
tiny_model.train()

for step in range(61):
    _, loss = tiny_model(x_batch, y_batch)   # 1. forward
    loss.backward()                          # 2. backward
    opt.step()                               # 3. update
    opt.zero_grad()                          # 4. reset
    if step % 10 == 0:
        print(f'step {step:3d}  loss {loss.item():.3f}')

del tiny_model, opt

## 7. Train for real

The real loop in `train.py` does the same four steps for 10,000 steps, with batches of 32 conversations, and adds a few things:

- **Learning rate schedule.** The learning rate is how big each nudge is. It starts at 0, climbs for 200 steps (**warmup**), then slowly falls along a cosine curve. Big steps early, careful steps late.
- **Gradient clipping.** If a gradient is huge, it is scaled down so one bad batch cannot wreck the model.
- **Mixed precision.** On a GPU, many calculations use 16-bit numbers instead of 32-bit. That is faster and uses less memory.
- **Evaluation.** Every 200 steps the model is tested on data it has never trained on. The best checkpoint is saved. If the training loss keeps falling while the eval loss rises, the model is memorising instead of learning.

Here is the learning rate schedule:

In [ ]:
import matplotlib.pyplot as plt
from config import TrainConfig
from train import get_lr

tc = TrainConfig()
steps = list(range(tc.max_steps))
plt.figure(figsize=(7, 3))
plt.plot(steps, [get_lr(s, tc) for s in steps])
plt.xlabel('step'); plt.ylabel('learning rate'); plt.title('Warmup, then cosine decay')
plt.show()

Now run the training. It takes about 4.5 minutes on an L4. It prints a row every 100 steps. Watch the `Train` column fall from about 8.3 to about 0.4, and watch the `Eval` column follow it.

In [ ]:
del model          # free the GPU memory used by the untrained model from step 5
from train import train

t0 = time.time()
train()
print(f'Total wall time: {time.time() - t0:.0f} s')

Training saved three kinds of files in `checkpoints/`: `best_model.pt` (lowest eval loss), `step_*.pt` (snapshots) and `final_model.pt` (the last step, with the full loss history). Let's plot the history.

In [ ]:
ckpt = torch.load('checkpoints/final_model.pt', map_location='cpu', weights_only=False)
losses = ckpt['train_losses']

def smooth(values, k=100):
    return [sum(values[max(0, i - k + 1): i + 1]) / len(values[max(0, i - k + 1): i + 1]) for i in range(len(values))]

plt.figure(figsize=(7, 3.5))
plt.plot(losses, alpha=0.25, label='per step')
plt.plot(smooth(losses), label='average of last 100 steps')
plt.xlabel('step'); plt.ylabel('training loss'); plt.legend(); plt.title('Training loss')
plt.show()
print(f'First loss: {losses[0]:.3f}   Last loss: {losses[-1]:.3f}')

The loss drops steeply in the first 1,000 steps, then flattens around 0.4. It does not reach 0 most likely because the same kind of question has many different valid replies in the data, so the model cannot predict the exact next word every time. It only needs to give good odds to each valid one.

## 8. Talk to Guppy

To generate text, the model runs in a loop:

1. Feed in the tokens so far.
2. Take the scores for the **next** token and turn them into probabilities with softmax.
3. Pick one token, append it, and go back to step 1.
4. Stop when the model writes `<|im_end|>`.

Let's load the best checkpoint and chat.

In [ ]:
from inference import GuppyInference

engine = GuppyInference('checkpoints/best_model.pt', 'data/tokenizer.json', device=device)

def chat(text, temperature=0.7):
    r = engine.chat_completion([{'role': 'user', 'content': text}], temperature=temperature, max_tokens=64)
    return r['choices'][0]['message']['content']

for q in ['hi guppy', 'are you hungry', 'do you like bubbles', 'what is the meaning of life', 'tell me a joke', 'goodnight guppy']:
    print(f'You>   {q}')
    print(f'Guppy> {chat(q)}\n')

**What does the model actually "think"?** At each step it gives a probability to every token in its vocabulary. Here are its five favourite choices for the first word of the reply:

In [ ]:
prompt = '<|im_start|>user\nare you hungry<|im_end|>\n<|im_start|>assistant\n'
x = torch.tensor([tokenizer.encode(prompt).ids], device=device)

engine.model.eval()
with torch.no_grad():
    logits, _ = engine.model(x)
probs = F.softmax(logits[0, -1], dim=-1)
top = probs.topk(5)
for p, i in zip(top.values, top.indices):
    print(f'{p.item()*100:5.1f}%  {tokenizer.decode([i.item()])!r}')

**Temperature** controls how adventurous the sampling is. The model divides its scores by the temperature before softmax:

- Low (0.1): the likeliest token almost always wins. Safe and repetitive.
- Medium (0.7): a good mix. This is the default.
- High (1.5 and up): unlikely tokens get a real chance. More surprising, and eventually nonsense.

Run the cell a few times. The 0.1 answers barely change. This model is small and its data is narrow, so even 1.5 stays fairly readable. Try 3.0 to see it fall apart.

In [ ]:
question = 'what do you think about the rain'   # <- try your own question

for temp in [0.1, 0.7, 1.5, 3.0]:
    print(f'--- temperature {temp} ---')
    for _ in range(3):
        print('  ', chat(question, temperature=temp))

### Chat with Guppy yourself

Type a message in the box below and press **Enter**. Guppy answers underneath. Each message is separate: Guppy does not remember earlier ones.
Move the slider to change the temperature and see how the replies change. Ideas to try: "what is your name", "do you like cats", "are you scared of the filter", "what is bitcoin".

In [ ]:
import ipywidgets as widgets
from IPython.display import display

box = widgets.Text(placeholder='say something to Guppy, then press Enter', layout=widgets.Layout(width='520px'))
temp_slider = widgets.FloatSlider(value=0.7, min=0.1, max=2.0, step=0.1, description='temperature', style={'description_width': 'initial'})
log = widgets.Output()

def on_send(_):
    text = box.value.strip()
    if not text:
        return
    box.value = ''
    with log:
        print(f'You>   {text}')
        print(f'Guppy> {chat(text, temperature=temp_slider.value)}\n')

box.on_submit(on_send)
display(box, temp_slider, log)

## 9. Where to go next

You trained a language model from raw text. The only difference between this and a large model is scale: more data, more parameters, more GPUs. Here are experiments to try. For each one, edit the file, retrain, and compare the loss curve and the replies.

1. **Train less.** In `config.py`, set `max_steps` to 1000. How much worse is Guppy? Where does the eval loss stop improving?
2. **Make the model smaller.** Set `n_layers` to 2 and `d_model` to 128. Check the parameter count and the quality.
3. **Break it on purpose.** Set `dropout` to 0 and `weight_decay` to 0, and train for 10,000 steps. Does the eval loss start rising while the train loss keeps falling?
4. **Change the learning rate.** Try `learning_rate = 3e-2` (100x too big). What happens to the loss?
5. **Ask it something it never saw.** "What is bitcoin?" Guppy was trained on fish topics only, so see what it makes up.

To edit a file in Colab, open the folder icon on the left, go to `guppy/`, and double-click the file. Do not re-run the `%%writefile` cells afterwards, because they would overwrite your edit.

Training writes into `checkpoints/`. To keep your first model for comparison, rename the folder first, then train again in a fresh process so it picks up your edit:

```
!mv checkpoints checkpoints_run1
!python train.py
```

**Keep your model.** Colab deletes everything in `/content` when the runtime disconnects or times out, and the trained model goes with it. The cell below packs the files you need into one archive. Uncomment the last two lines to download it to your computer.

In [ ]:
!cd /content && tar czf guppylm.tar.gz guppy/checkpoints/best_model.pt guppy/checkpoints/config.json guppy/data/tokenizer.json guppy/model.py guppy/config.py guppy/inference.py
print(f'Archive size: {os.path.getsize("/content/guppylm.tar.gz") / 1e6:.1f} MB')

# from google.colab import files
# files.download('/content/guppylm.tar.gz')